<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/3%20-%20Regress%C3%A3o%20Log%C3%ADstica/3.4_regressao_softmax_mnist_minibatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 3.4 — Regressão softmax no MNIST com mini-batches

## Objetivos

- Treinar o mesmo modelo do notebook 3.2, `nn.Linear(784, 10)`, agora com as 60 mil imagens de treino do MNIST.
- Usar `Dataset` e `DataLoader` (notebook 3.3) para o gradiente descendente estocástico por mini-batch.
- Separar treino e validação com `random_split`.
- Usar `model.train()` e `model.eval()`.
- Acompanhar o custo de cada mini-batch (épocas fracionárias) e o custo e a acurácia por época.

## O problema

A tarefa é a mesma do 3.2: reconhecer o dígito (0 a 9) numa imagem 28 × 28. No 3.2 usamos 1000 imagens, todas juntas em cada época. Agora usamos todo o conjunto de treino. Em vez de calcular o gradiente com todas as amostras a cada passo, calculamos com um **mini-batch** de 100 amostras e atualizamos os parâmetros logo em seguida: é o gradiente descendente estocástico (SGD) por mini-batch. Com 48 mil imagens de treino, cada época passa a ter 480 atualizações, em vez de uma.

Nos gráficos de custo por mini-batch, o eixo horizontal está em épocas fracionárias: o mini-batch 240 da primeira época aparece em 0,5.

## Importação das bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import MNIST

torch.manual_seed(0)
np.random.seed(0)

# Usa a GPU se houver uma disponível (no Colab: Ambiente de execução → Alterar o tipo de ambiente de execução)
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## Dataset e DataLoader

### Definição do tamanho do mini-batch

In [ ]:
batch_size = 100

### Carregamento, divisão treino/validação e criação dos DataLoaders

- `MNIST(root='./data', ...)` usa o mesmo diretório do notebook 3.2: se os arquivos já foram baixados, não baixa de novo.
- `transform=ToTensor()` é aplicado a cada imagem quando o dataset é indexado: converte a imagem (pixels inteiros de 0 a 255) num tensor `float32` de shape (1, 28, 28) com valores entre 0 e 1, a mesma normalização feita à mão no 3.2.
- `random_split` sorteia 80% das 60 mil imagens para treino e 20% para validação. O conjunto de teste do MNIST (`train=False`) fica reservado.
- Só o DataLoader de treino usa `shuffle=True`; a ordem da validação não afeta o resultado.

In [ ]:
full_train_dataset = MNIST(root='./data', train=True, download=True,
                           transform=torchvision.transforms.ToTensor())

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

# generator com semente: a divisão é a mesma a cada execução
dataset_train, dataset_val = torch.utils.data.random_split(
    full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(0))

loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

print(f'Amostras no conjunto de treino original do MNIST: {len(full_train_dataset)}')
print(f'Amostras de treinamento: {len(dataset_train)}')
print(f'Amostras de validação:   {len(dataset_val)}')
print('Mini-batches de treinamento por época:', len(loader_train))
print('Mini-batches de validação:            ', len(loader_val))

x_train_sample, y_train_sample = next(iter(loader_train))  # primeiro mini-batch
print("\nDimensões das imagens de um mini-batch:", x_train_sample.size())
print("Dimensões dos rótulos de um mini-batch:", y_train_sample.size())
print("Valores mínimo e máximo dos pixels:    ", torch.min(x_train_sample).item(), torch.max(x_train_sample).item())
print("Tipo dos dados das imagens:            ", x_train_sample.dtype)
print("Tipo das classes das imagens:          ", y_train_sample.dtype)

### Visualizando um mini-batch

In [ ]:
grid = torchvision.utils.make_grid(x_train_sample[:24], pad_value=1.0, padding=1, nrow=12)
plt.figure(figsize=(15, 3))
plt.imshow(grid.numpy().transpose(1, 2, 0))
plt.axis('off')
plt.show()
print('classes:', y_train_sample[:24].view(2, 12).tolist())

## Modelo

O modelo é o mesmo do 3.2. `.to(device)` move os parâmetros para a GPU, se houver; os dados de cada mini-batch também precisam ser movidos para o mesmo `device`.

In [ ]:
model = nn.Linear(28*28, 10).to(device)  # 784 entradas, 10 saídas: 7.850 parâmetros

Testando com o primeiro mini-batch: as imagens chegam com shape (100, 1, 28, 28) e `view(-1, 28*28)` as transforma em (100, 784). A saída são 100 × 10 logits.

In [ ]:
x = x_train_sample.view(-1, 28*28).to(device)
print(x.shape)
logits = model(x)
print(logits.shape)

## Treinamento

### Inicialização dos hiperparâmetros, da função de custo e do otimizador

In [ ]:
n_epochs = 5
lr = 0.5

# CrossEntropyLoss como função de custo: recebe os logits (sem softmax) e os índices das classes
criterion = nn.CrossEntropyLoss()

# Gradiente descendente: a cada mini-batch, W ← W − lr·dJ/dW
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

### Laço de treinamento dos parâmetros

São dois laços: o externo percorre as épocas e o interno percorre os mini-batches do `loader_train`. Cada passo do laço interno é uma iteração: forward, custo, `zero_grad()`, `backward()` e `step()` com 100 amostras.

- `model.train()` e `model.eval()` alternam o modelo entre os modos de treino e de avaliação. Para `nn.Linear` não há diferença, mas camadas como dropout e batch normalization (módulo 4) se comportam de forma diferente em cada modo; por isso usamos sempre esse padrão.
- A validação roda dentro de `torch.no_grad()`, sem calcular gradientes.
- O custo de cada época é a média dos custos dos mini-batches. Além disso, guardamos o custo de cada mini-batch em `losses`, com a época fracionária correspondente em `epochs_frac`.

In [ ]:
train_losses_per_epoch = []
val_losses_per_epoch = []
train_accuracies_per_epoch = []
val_accuracies_per_epoch = []
losses = []       # custo de cada mini-batch de treino
epochs_frac = []  # época fracionária de cada mini-batch

for epoch in range(n_epochs):
    # Treino
    model.train()
    correct_train = 0
    total_train = 0
    running_train_loss = 0.0

    for i, (x_batch, y_batch) in enumerate(loader_train):
        inputs = x_batch.view(-1, 28 * 28).to(device)  # (100, 784)
        labels = y_batch.to(device)                    # (100,)

        outputs = model(inputs)                        # logits (100, 10)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        epochs_frac.append(epoch + (i + 1) / len(loader_train))

        running_train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    avg_train_loss = running_train_loss / len(loader_train)
    train_accuracy = correct_train / total_train
    train_losses_per_epoch.append(avg_train_loss)
    train_accuracies_per_epoch.append(train_accuracy)

    # Validação
    model.eval()
    correct_val = 0
    total_val = 0
    running_val_loss = 0.0

    with torch.no_grad():
        for x_batch, y_batch in loader_val:
            inputs_val = x_batch.view(-1, 28 * 28).to(device)
            labels_val = y_batch.to(device)

            outputs_val = model(inputs_val)
            loss_val = criterion(outputs_val, labels_val)

            running_val_loss += loss_val.item()
            predicted_val = outputs_val.argmax(dim=1)
            total_val += labels_val.size(0)
            correct_val += (predicted_val == labels_val).sum().item()

    avg_val_loss = running_val_loss / len(loader_val)
    val_accuracy = correct_val / total_val
    val_losses_per_epoch.append(avg_val_loss)
    val_accuracies_per_epoch.append(val_accuracy)

    print(f'Época [{epoch+1}/{n_epochs}], '
          f'custo treino: {avg_train_loss:.4f}, acurácia treino: {train_accuracy:.4f}, '
          f'custo validação: {avg_val_loss:.4f}, acurácia validação: {val_accuracy:.4f}')

O custo e a acurácia de treino de cada época são médias calculadas durante a época, enquanto os parâmetros ainda mudavam; os de validação são calculados com os parâmetros do fim da época. Por isso, na primeira época, a validação pode sair melhor que o treino.

In [ ]:
print('--- Treinamento finalizado ---')
print(f'Último custo de treinamento:     {train_losses_per_epoch[-1]:.4f}')
print(f'Última acurácia de treinamento:  {train_accuracies_per_epoch[-1]:.4f}')
print(f'Último custo de validação:       {val_losses_per_epoch[-1]:.4f}')
print(f'Última acurácia de validação:    {val_accuracies_per_epoch[-1]:.4f}')

### Visualizando a evolução do custo e da acurácia

À esquerda, o custo de cada mini-batch (em azul claro) e os custos médios por época. O custo de um mini-batch oscila porque cada um tem amostras diferentes; a tendência é de queda. À direita, a acurácia por época.

In [ ]:
epochs_axis = range(1, n_epochs + 1)
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_frac, losses, color='tab:blue', alpha=0.3, linewidth=0.8, label='custo por mini-batch (treino)')
plt.plot(epochs_axis, train_losses_per_epoch, 'o-', color='tab:blue', label='custo médio da época (treino)')
plt.plot(epochs_axis, val_losses_per_epoch, 's-', color='tab:orange', label='custo de validação')
plt.xlabel('época')
plt.ylabel('custo (cross-entropy)')
plt.ylim(0, 1.0)
plt.title('Evolução do custo')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_axis, train_accuracies_per_epoch, 'o-', label='treino')
plt.plot(epochs_axis, val_accuracies_per_epoch, 's-', label='validação')
plt.xlabel('época')
plt.ylabel('acurácia')
plt.title('Evolução da acurácia')
plt.xticks(epochs_axis)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Observações sobre os gráficos

Custo e acurácia de treino e de validação ficam próximos: com 48 mil imagens de treino, o modelo linear não tem parâmetros suficientes para decorar os dados, diferente do 3.2, em que havia só 1000 imagens. Já na primeira época o custo cai para perto do valor final, porque essa época tem 480 atualizações dos parâmetros. Uma diferença grande entre as curvas de treino e de validação indicaria sobreajuste (overfitting), tema do notebook 4.3.

## Avaliação

### Pesos de cada classe

Como no 3.2, cada linha de $W$ pode ser vista como uma imagem 28 × 28. Compare com as imagens de pesos do 3.2, treinadas com 1000 imagens.

In [ ]:
weights = model.weight.detach().cpu()  # .detach(): tensor sem histórico de gradientes; .cpu(): traz da GPU, se for o caso
grid = torchvision.utils.make_grid(weights.view(-1, 1, 28, 28), normalize=True, scale_each=True,
                                   pad_value=1.0, padding=1, nrow=10)
plt.figure(figsize=(15, 2.5))
plt.imshow(grid.numpy().transpose(1, 2, 0))
plt.title('pesos das classes 0 a 9')
plt.axis('off')
plt.show()

## Exercícios

### Perguntas

1. Qual é o tamanho do mini-batch?
2. Em uma época, quantos mini-batches existem?
3. Qual é a definição de época?

*Escreva sua resposta aqui.*

### Exercícios

1. Treine um modelo novo e imprima uma linha a cada 100 mini-batches (e no último mini-batch da época), no mesmo estilo do print de final de época, por exemplo:
    - `Época: 1/5, batch: 100/480, custo: 0.3512`

   Para não imprimir 480 linhas por época, a cada 100 mini-batches basta.

In [ ]:
torch.manual_seed(0)
model_ex = nn.Linear(28*28, 10).to(device)
optimizer_ex = torch.optim.SGD(model_ex.parameters(), lr=lr)

for epoch in range(n_epochs):
    model_ex.train()
    for i, (x_batch, y_batch) in enumerate(loader_train):
        inputs = x_batch.view(-1, 28 * 28).to(device)
        labels = y_batch.to(device)
        loss = criterion(model_ex(inputs), labels)
        optimizer_ex.zero_grad()
        loss.backward()
        optimizer_ex.step()

        # TODO: escreva sua solução aqui


2. Altere o tamanho do mini-batch (`batch_size`) algumas vezes (por exemplo 10, 100 e 1000), refaça o treinamento com um modelo novo e o mesmo `lr`, e compare num gráfico a queda do custo por mini-batch em função da época fracionária. Para ser rápido, use 1 época para cada tamanho. Meça também o tempo de cada época.

In [ ]:
import time

# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- O SGD por mini-batch atualiza os parâmetros depois de cada mini-batch: com 48 mil amostras e `batch_size = 100`, uma época tem 480 iterações.
- O custo de cada mini-batch oscila, porque cada mini-batch tem amostras diferentes; a média por época mostra a tendência.
- `batch_size` é um hiperparâmetro: mini-batches menores dão mais atualizações por época, com gradientes mais ruidosos; maiores dão menos atualizações, mais estáveis.
- Com 48 mil imagens de treino, as curvas de treino e de validação ficam próximas e a acurácia de validação passa de 0,84 (3.2) para cerca de 0,92.
- O padrão `model.train()` no treino e `model.eval()` + `torch.no_grad()` na avaliação será usado em todos os notebooks seguintes.